# Preprocess dinoflagellate single-cell expression data

Standalone script for manuscript code availability. Takes the raw
dinoflagellate AnnData object produced by `subset_dinoflagellate_data.ipynb`
and applies quality control, normalization, scaling, PCA, and nonlinear
embeddings (UMAP and t-SNE) using Scanpy v1.9.3.

Input: `Alps_Dinoflagellata.h5ad` (422 cells, raw counts)

Output: `Alps_preprocessed_Dinoflagellata.h5ad` (372 cells after QC)

No external project modules are required beyond standard Python packages.

In [1]:
import os
from pathlib import Path
from typing import Iterable, Optional, Sequence, Set, Tuple, Union

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.stats import median_abs_deviation as mad

In [2]:
def read_emapper_annotations(path: Union[str, Path]) -> pd.DataFrame:
    """Load eggNOG-mapper annotation file (.emapper.annotations)."""
    header = None
    skiprows = []
    with Path(path).open() as handle:
        for idx, line in enumerate(handle):
            if line.startswith("##"):
                skiprows.append(idx)
            elif line.startswith("#query"):
                header = line.lstrip("#").strip().split("\t")
                skiprows.append(idx)
                break
    if header is None:
        raise ValueError(f"No #query header line found in {path}")
    return pd.read_csv(
        path,
        sep="\t",
        names=header,
        skiprows=skiprows,
        dtype=str,
        low_memory=False,
    )


def build_mito_ribo_from_eggnog(
    eggnog_df: pd.DataFrame,
    id_col: str = "query",
) -> Tuple[Set[str], Set[str]]:
    """Identify mitochondrial and ribosomal genes from eggNOG annotations."""
    df = eggnog_df.copy()

    def col(name: str) -> pd.Series:
        return df[name].fillna("").astype(str) if name in df.columns else pd.Series([""] * len(df))

    text = (
        col("Description")
        + " "
        + col("Preferred_name")
        + " "
        + col("PFAMs")
        + " "
        + col("eggNOG_OGs")
    ).str.lower()
    cog = col("COG_category")

    ribo_kw = r"(?:ribosomal|rpl\d*|rps\d*|\b60s\b|\b40s\b|\b50s\b|\b30s\b)"
    is_ribo = cog.str.contains("J", na=False) | text.str.contains(ribo_kw, regex=True)

    mito_kw = (
        r"(?:mitochond|respirat|oxidative phosphorylation|electron transport|"
        r"etc complex|cytochrome c oxidase|nad[h]? dehydrogenase|atp synthase)"
    )
    is_mito = text.str.contains(mito_kw, regex=True)

    ribo_genes = set(df.loc[is_ribo, id_col].astype(str))
    mito_genes = set(df.loc[is_mito, id_col].astype(str))
    return mito_genes, ribo_genes


def annotate_mito_ribo(
    adata,
    eggnog_df: pd.DataFrame,
) -> None:
    """Add `is_mito` and `is_ribo` columns to `.var`."""
    present = eggnog_df[eggnog_df["query"].isin(adata.var_names)]
    mito_genes, ribo_genes = build_mito_ribo_from_eggnog(present, id_col="query")
    adata.var["is_mito"] = adata.var_names.isin(mito_genes)
    adata.var["is_ribo"] = adata.var_names.isin(ribo_genes)


def _total_counts(adata) -> np.ndarray:
    x = adata.X
    return np.asarray(x.sum(axis=1)).ravel() if sparse.issparse(x) else np.asarray(x.sum(axis=1)).ravel()


def ensure_raw_counts_layer(adata) -> None:
    """Ensure `.X` and `.layers['counts']` both hold raw counts."""
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    else:
        adata.layers["counts"] = adata.X.copy()


def filter_cells_by_library_size(
    adata,
    low_percentile: float = 1.0,
    high_percentile: float = 99.0,
):
    """Remove cells in the lowest/highest UMI percentiles (doublet/debris control)."""
    total_counts = _total_counts(adata)
    low, high = np.percentile(total_counts, [low_percentile, high_percentile])
    keep = (total_counts >= low) & (total_counts <= high)
    removed = int((~keep).sum())
    print(f"Library-size filter ({low_percentile}–{high_percentile} percentiles): removed {removed} cells")
    return adata[keep].copy()


def filter_high_mitochondrial_cells(
    adata,
    method: str = "mad",
    fixed_cutoff: float = 15.0,
    nmads: float = 5.0,
    cap: Optional[float] = 20.0,
):
    """Remove cells with high mitochondrial read fraction.

    Must be run on raw counts before mitochondrial genes are dropped.
    """
    if "is_mito" not in adata.var.columns:
        raise KeyError("Run annotate_mito_ribo() before mitochondrial filtering.")

    sc.pp.calculate_qc_metrics(adata, qc_vars=["is_mito"], layer="counts", inplace=True)
    mt = adata.obs["pct_counts_is_mito"].to_numpy()

    if method == "fixed":
        cutoff = float(fixed_cutoff)
    elif method == "mad":
        cutoff = float(np.median(mt) + nmads * mad(mt))
        if cap is not None:
            cutoff = min(cutoff, float(cap))
    else:
        raise ValueError("method must be 'mad' or 'fixed'")

    adata.obs["high_mito"] = mt > cutoff
    removed = int(adata.obs["high_mito"].sum())
    print(f"Mitochondrial filter (>{cutoff:.2f}%): removed {removed} cells")
    return adata[~adata.obs["high_mito"]].copy()


def drop_mito_ribo_genes(adata):
    """Permanently remove mitochondrial and ribosomal genes."""
    for key in ("is_mito", "is_ribo"):
        if key not in adata.var.columns:
            raise KeyError(f"Missing ad.var['{key}']")
    exclude = adata.var[["is_mito", "is_ribo"]].any(axis=1).to_numpy()
    adata = adata[:, ~exclude].copy()
    adata.layers["counts"] = adata.layers["counts"].copy()
    return adata


def preprocess_expression(
    adata,
    target_sum: float = 1e4,
    n_pcs: int = 30,
    scale_max_value: float = 10.0,
    random_state: int = 42,
):
    """Normalize, log-transform, scale, and compute PCA on all remaining genes."""
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    adata.uns["log1p"] = {"base": float(np.e)}

    adata.layers["lognorm"] = adata.X.copy()

    if "sample" in adata.obs:
        adata.obs["sample"] = adata.obs["sample"].astype("category")

    sc.pp.scale(adata, max_value=scale_max_value)
    sc.tl.pca(adata, n_comps=n_pcs, svd_solver="arpack", random_state=random_state)
    return adata


def compute_embeddings(
    adata,
    n_neighbors: int = 15,
    umap_min_dist: float = 0.8,
    umap_spread: float = 2.0,
    tsne_perplexity: float = 60.0,
    tsne_learning_rate: float = 500.0,
    random_state: int = 42,
):
    """Build the neighbor graph, UMAP, and t-SNE from the PCA representation."""
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep="X_pca")
    sc.tl.umap(
        adata,
        min_dist=umap_min_dist,
        spread=umap_spread,
        random_state=random_state,
    )
    sc.tl.tsne(
        adata,
        use_rep="X_pca",
        perplexity=tsne_perplexity,
        learning_rate=tsne_learning_rate,
        random_state=random_state,
    )
    return adata

In [3]:
# --- configuration ---
base_path = Path("/home/labs/vardi/amirz/pairing/Alps_All_MetaG")
input_h5ad = base_path / "Alps_Dinoflagellata.h5ad"
output_h5ad = base_path / "Alps_preprocessed_Dinoflagellata.h5ad"

eggnog_file = (
    "/home/labs/vardi/Collaboration/databases/Gossenkollesee/"
    "MetaG_eggnog/MetaG_eggnog.emapper.annotations"
)

# QC
min_gene_counts = 2
library_low_pct = 1.0
library_high_pct = 99.0
mito_method = "mad"  # adaptive cutoff; use "fixed" with mito_fixed_cutoff for a set threshold
mito_nmads = 5.0
mito_cap = 20.0
mito_fixed_cutoff = 15.0

# normalization / dimensionality reduction
target_sum = 1e4
n_pcs = 30
scale_max_value = 10.0
n_neighbors = 15
umap_min_dist = 0.8
umap_spread = 2.0
tsne_perplexity = 60.0
tsne_learning_rate = 500.0
random_state = 42

### Load raw dinoflagellate data and eggNOG annotations

In [4]:
adata = sc.read_h5ad(input_h5ad)
eggnog_df = read_emapper_annotations(eggnog_file)
print(f"Loaded {adata.n_obs} cells × {adata.n_vars} genes from {input_h5ad.name}")
adata

Loaded 422 cells × 8272 genes from Alps_Dinoflagellata.h5ad


AnnData object with n_obs × n_vars = 422 × 8272
    obs: 'sample', 'n_counts'
    var: 'gene_ids', 'feature_types', 'n_counts'
    layers: 'counts'

### Gene filtering and quality control

In [5]:
ensure_raw_counts_layer(adata)
sc.pp.filter_genes(adata, min_counts=min_gene_counts)
print(f"After gene filter (min_counts={min_gene_counts}): {adata.n_obs} cells × {adata.n_vars} genes")

annotate_mito_ribo(adata, eggnog_df)
print(
    f"Annotated {adata.var['is_mito'].sum()} mitochondrial and "
    f"{adata.var['is_ribo'].sum()} ribosomal genes"
)

adata = filter_cells_by_library_size(
    adata,
    low_percentile=library_low_pct,
    high_percentile=library_high_pct,
)
adata = filter_high_mitochondrial_cells(
    adata,
    method=mito_method,
    fixed_cutoff=mito_fixed_cutoff,
    nmads=mito_nmads,
    cap=mito_cap,
)
adata = drop_mito_ribo_genes(adata)
print(f"After QC: {adata.n_obs} cells × {adata.n_vars} genes")
adata

After gene filter (min_counts=2): 422 cells × 8272 genes
Annotated 272 mitochondrial and 761 ribosomal genes
Library-size filter (1.0–99.0 percentiles): removed 10 cells
Mitochondrial filter (>11.98%): removed 40 cells
After QC: 372 cells × 7248 genes


AnnData object with n_obs × n_vars = 372 × 7248
    obs: 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_is_mito', 'log1p_total_counts_is_mito', 'pct_counts_is_mito', 'high_mito'
    var: 'gene_ids', 'feature_types', 'n_counts', 'is_mito', 'is_ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    layers: 'counts'

### Normalization, scaling, PCA, and embeddings

In [6]:
adata = preprocess_expression(
    adata,
    target_sum=target_sum,
    n_pcs=n_pcs,
    scale_max_value=scale_max_value,
    random_state=random_state,
)
adata = compute_embeddings(
    adata,
    n_neighbors=n_neighbors,
    umap_min_dist=umap_min_dist,
    umap_spread=umap_spread,
    tsne_perplexity=tsne_perplexity,
    tsne_learning_rate=tsne_learning_rate,
    random_state=random_state,
)
adata

/home/labs/vardi/amirz/.conda/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 372 × 7248
    obs: 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_is_mito', 'log1p_total_counts_is_mito', 'pct_counts_is_mito', 'high_mito'
    var: 'gene_ids', 'feature_types', 'n_counts', 'is_mito', 'is_ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mean', 'std'
    uns: 'log1p', 'pca', 'neighbors', 'umap', 'tsne'
    obsm: 'X_pca', 'X_umap', 'X_tsne'
    varm: 'PCs'
    layers: 'counts', 'lognorm'
    obsp: 'distances', 'connectivities'

### Save preprocessed object

In [7]:
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata.write(output_h5ad)
print(f"Saved {adata.n_obs} cells × {adata.n_vars} genes to {output_h5ad}")
print("log1p metadata:", adata.uns.get("log1p"))

Saved 372 cells × 7248 genes to /home/labs/vardi/amirz/pairing/Alps_All_MetaG/Alps_preprocessed_Dinoflagellata.h5ad
log1p metadata: {'base': 2.718281828459045}
